# 07 - Cleaning Eurostat Citizenship Acquisition for Cameroonian Citizens

This notebook cleans Eurostat data on citizenship acquisition by Cameroonian citizens in European destination countries.

Analytical role:

- supports Q3 on post-arrival trajectories
- tracks naturalization-related outcomes across available reference years
- standardizes citizenship acquisition indicators for `post_arrival_master.csv`

Citizenship acquisition is not interpreted as the same phenomenon as first permits, long-term residence or status changes.


## Output Preparation

The cleaned dataset is reshaped into the project-wide standardized format. This makes it possible to compare countries and years while keeping the indicator definition explicit through `measure_type`.


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

EUROPE_PATH = DATA_RAW / "europe"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_citizenship_acquisition_cameroon.xlsx")
xls.sheet_names

In [ ]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_citizenship_acquisition_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.drop(columns=[2, 4, 6, 8, 10, 12, 14, 16, 18], errors="ignore")
sheet1_raw = sheet1_raw.drop(index=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 57, 58, 59, 60, 61, 62, 63, 64], errors="ignore").reset_index(drop=True)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.drop(index=[41], errors="ignore").reset_index(drop=True)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.replace(":", 0)
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)

sheet1_raw

In [ ]:
# Use the first row as the header
sheet1_raw.columns = sheet1_raw.iloc[0]
sheet1_raw = sheet1_raw.iloc[1:].reset_index(drop=True)

# Rename TIME to destination_country
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)

# Remove the GEO (Labels) row if it is still present
sheet1_raw = sheet1_raw[
    sheet1_raw["destination_country"] != "GEO (Labels)"
]

# Reshape from wide format to long format
sheet1_raw = sheet1_raw.melt(
    id_vars=["destination_country"],
    var_name="year",
    value_name="permit"
)

# Clean data types
sheet1_raw["year"] = sheet1_raw["year"].astype(int)

sheet1_raw["permit"] = (
    pd.to_numeric(sheet1_raw["permit"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# Reorder columns
sheet1_raw = sheet1_raw[
    ["destination_country", "year", "permit"]
]

sheet1_raw

In [ ]:
# Rename permit to permits
sheet1_raw = sheet1_raw.rename(
    columns={"permit": "permits"}
)

# Add source and nature columns
sheet1_raw["source"] = "eurostat"
sheet1_raw["nature"] = "citizenship_acquisition"

# Add the operational reference period
def classify_covid_period(year):
    if year < 2020:
        return "pre_covid"
    elif year in [2020, 2021]:
        return "covid"
    else:
        return "post_covid"

sheet1_raw["covid_period"] = sheet1_raw["year"].apply(classify_covid_period)

# Reorder columns
sheet1_raw = sheet1_raw[
    [
        "destination_country",
        "year",
        "permits",
        "source",
        "nature",
        "covid_period"
    ]
]

sheet1_raw

In [ ]:
sheet1_raw.to_csv(
    CLEANED_PATH / "eurostat_citizenship_acquisition_cleaned.csv",
    index=False
)